# 03_build_indicators.py 결과 확인

`data/indicators`에 저장된 지표 계산 결과를 확인한다.
- 전체 건수, 컬럼별 결측 비율
- 종목별 지표 스냅샷 (EPS/BPS/PER/PBR/ROE/부채비율/배당수익률/ROA/모멘텀/F-Score/EPS성장률)
- 이상치(완전자본잠식 등 N/A 처리된 종목) 확인

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.master("spark://spark-master:7077").appName("check_indicators").getOrCreate()

df = spark.read.parquet("/opt/spark-apps/data/indicators")
total = df.count()
print(f"전체 {total}건")
df.printSchema()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/04 08:24:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


전체 18598건
root
 |-- stock_code: string (nullable = true)
 |-- fs_div: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- rcept_no: string (nullable = true)
 |-- eps: double (nullable = true)
 |-- bps: double (nullable = true)
 |-- per: double (nullable = true)
 |-- pbr: double (nullable = true)
 |-- roe: double (nullable = true)
 |-- debt_ratio: double (nullable = true)
 |-- dividend_yield: double (nullable = true)
 |-- roa: double (nullable = true)
 |-- momentum: double (nullable = true)
 |-- f_score: integer (nullable = true)
 |-- eps_growth_rate: double (nullable = true)
 |-- net_income_krw: long (nullable = true)
 |-- equity_krw: long (nullable = true)
 |-- dividend_amount: long (nullable = true)
 |-- year: integer (nullable = true)



## 1. 종목별 지표 스냅샷 (표로 보기)

In [2]:
cols = ["stock_code", "eps", "bps", "per", "pbr", "roe", "debt_ratio",
        "dividend_yield", "roa", "momentum", "f_score", "eps_growth_rate"]

pdf = df.select(*cols).orderBy("stock_code").toPandas()
pdf

,stock_code,eps,bps,per,pbr,roe,debt_ratio,dividend_yield,roa,momentum,f_score,eps_growth_rate
0,000020,773.01,13557.23,6.18,0.35,5.70,21.78,3.77,4.68,-26.81,8.0,10.26
1,000020,701.10,12893.46,6.82,0.37,5.44,24.34,3.77,4.37,-26.81,8.0,-31.80
2,000020,1010.94,14573.40,4.73,0.33,6.94,38.79,3.77,5.00,-26.81,5.0,30.78
3,000020,1010.94,14573.40,4.73,0.33,6.94,38.79,3.77,5.00,-26.81,5.0,30.78
4,000020,701.10,12893.46,6.82,0.37,5.44,24.34,3.77,4.37,-26.81,8.0,-31.80
...,...,...,...,...,...,...,...,...,...,...,...,...
18593,950220,-1752.31,2486.69,NaN,0.53,-70.47,20.91,NaN,-58.28,3.20,2.0,NaN
18594,950220,-1964.30,5854.73,NaN,0.23,-33.55,15.42,NaN,-29.07,3.20,3.0,NaN
18595,950220,-2002.68,4045.34,NaN,0.33,-49.51,16.86,NaN,-42.36,3.20,2.0,NaN
18596,950220,-1105.49,3315.89,NaN,0.40,-33.34,13.56,NaN,-29.36,3.20,1.0,NaN


## 2. 컬럼별 결측 비율 (%)
PER/배당수익률/EPS성장률은 적자기업·전년도 재무제표 없음 등으로 원래도 일정 비율 NULL이 나오는 게 정상.

In [3]:
metric_cols = ["eps", "bps", "per", "pbr", "roe", "debt_ratio",
               "dividend_yield", "roa", "momentum", "f_score", "eps_growth_rate"]

null_ratio = df.select([
    F.round(F.sum(F.col(c).isNull().cast("int")) / F.count("*") * 100, 1).alias(c)
    for c in metric_cols
])
null_ratio.toPandas()

,eps,bps,per,pbr,roe,debt_ratio,dividend_yield,roa,momentum,f_score,eps_growth_rate
0,0.0,1.4,34.2,1.4,1.4,1.4,60.3,0.4,0.1,2.8,35.5


## 3. 완전자본잠식 종목 (equity<=0 이라 BPS/PBR/ROE/부채비율이 전부 N/A인 케이스)
EPS는 있는데 BPS가 NULL이면 자본총계가 0 이하라는 뜻 (FinancialIndicatorService.calculate와 동일 조건).

In [4]:
capital_impaired = df.filter(F.col("eps").isNotNull() & F.col("bps").isNull())
print(f"완전자본잠식 추정 종목: {capital_impaired.count()}건")
capital_impaired.select(*cols).toPandas()

완전자본잠식 추정 종목: 261건


,stock_code,eps,bps,per,pbr,roe,debt_ratio,dividend_yield,roa,momentum,f_score,eps_growth_rate
0,376900,116.99,NaN,334.64,NaN,NaN,NaN,NaN,13.10,213.98,4.0,NaN
1,451760,-1254.26,NaN,NaN,NaN,NaN,NaN,NaN,-24.97,12.49,4.0,NaN
2,453340,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2.41,1.0,NaN
3,462350,-1429.84,NaN,NaN,NaN,NaN,NaN,NaN,-205.62,-49.46,2.0,NaN
4,462520,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.84,1.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
256,418420,-94.21,NaN,NaN,NaN,NaN,NaN,NaN,-64.30,-12.76,NaN,NaN
257,419080,37.60,NaN,115.03,NaN,NaN,NaN,NaN,2.48,-41.96,NaN,NaN
258,419530,-1022.55,NaN,NaN,NaN,NaN,NaN,NaN,-32.25,-68.87,NaN,NaN
259,441270,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-40.54,NaN,NaN


## 4. 지표 분포 요약 (min/max/mean 등) - 극단치 스캔용

In [5]:
df.select(*metric_cols).summary("min", "25%", "50%", "75%", "max").toPandas()

26/08/04 08:24:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,summary,eps,bps,per,pbr,roe,debt_ratio,dividend_yield,roa,momentum,f_score,eps_growth_rate
0,min,-506965.83,7.65,0.0,0.0,-4901.41,0.06,0.0,-7548.99,-93.53,0,-536694.69
1,25%,-152.27,2325.99,5.42,0.43,-4.52,31.77,1.18,-2.49,-33.05,4,-72.83
2,50%,196.77,5662.09,10.88,0.79,4.25,68.14,2.31,2.19,-12.29,5,-5.91
3,75%,944.85,13790.94,24.71,1.78,10.43,131.28,4.14,6.21,20.45,7,62.38
4,max,652892.74,2.03162906611E9,17159.09,1588.22,16245.01,5.82409234E7,97.3,8825.46,2900.0,9,207399.29


In [6]:
spark.stop()